# EBiEOT Classification on MNIST

Semi-supervised energy-based classification using `ClassificationBasedEBiEOT`,
aligned with `scripts/run_mnist_energy_grid.py` / `mnist_energy_grid.py`.

In [ ]:
import os

from src.utils.notebook_setup import ensure_repo_imports

REPO_ROOT = ensure_repo_imports()
os.chdir(REPO_ROOT)
print(f"Repo root: {REPO_ROOT}")

In [ ]:
import random

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

from src.ebieot.classification_based import ClassificationBasedEBiEOT
from src.utils.datasets.mnist_classification import (
    build_mnist_datasets,
    build_paired_marginal_val_indices,
    evaluate_classification,
    indices_by_class_from_dataset,
    run_classification_training,
    val_per_class_from_paired,
)

In [ ]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

pairedPerClass = 20
unpairedPerClass = 200
valRatio = 0.25

trainData, testData, numClasses, inChannels, imageSize = build_mnist_datasets(
    str(REPO_ROOT / "data" / "mnist")
)
byClass = indices_by_class_from_dataset(trainData, numClasses)
valPerClass = val_per_class_from_paired(pairedPerClass, valRatio)

rng = random.Random(seed + 10_000)
pairedIdx, marginalIdx, valIdx = build_paired_marginal_val_indices(
    byClass,
    paired_per_class=pairedPerClass,
    marginal_per_class=unpairedPerClass,
    val_per_class=valPerClass,
    rng=rng,
)
if len(marginalIdx) == 0:
    marginalIdx = pairedIdx.copy()

pinMem = torch.cuda.is_available()
pairedLoader = DataLoader(
    Subset(trainData, pairedIdx), batch_size=64, shuffle=True, pin_memory=pinMem
)
marginalLoader = DataLoader(
    Subset(trainData, marginalIdx), batch_size=128, shuffle=True, pin_memory=pinMem
)
valLoader = DataLoader(
    Subset(trainData, valIdx), batch_size=512, shuffle=False, pin_memory=pinMem
)
testLoader = DataLoader(testData, batch_size=512, shuffle=False, pin_memory=pinMem)

model = ClassificationBasedEBiEOT(
    num_classes=numClasses,
    in_channels=inChannels,
    image_size=imageSize,
    epsilon=0.5,
    arch="cnn",
    hidden_dim=128,
).to(device)

print("Data and model prepared")

### Semi-supervised likelihood

The model maximizes data likelihood on labeled pairs while using unlabeled \(x\) to sharpen the energy landscape. Class potentials add a low-dimensional term so posterior \(p(y|x)\) stays normalized over labels.

In [ ]:
epochsMax = 3  # increase for full runs (reference default: 25)

result = run_classification_training(
    model,
    pairedLoader,
    marginalLoader,
    valLoader,
    testLoader,
    device,
    epochs_max=epochsMax,
    patience=6,
    lr=1e-3,
    weight_decay=1e-4,
    grad_clip=5.0,
    ema_decay=0.999,
    use_ema=True,
)

for rec in result["epoch_records"]:
    print(
        f"epoch={rec['epoch']} train_loss={rec['train_loss']:.4f} "
        f"val_acc={rec['val_acc']:.4f}"
    )

print(
    f"best_val_acc={result['best_val_acc']:.4f} "
    f"test_acc={result['test_acc']:.4f} test_loss={result['test_loss']:.4f}"
)

In [ ]:
model.eval()
with torch.no_grad():
    xShow, yShow = next(iter(testLoader))
    xShow = xShow[:12].to(device)
    yShow = yShow[:12].to(device)
    pred = model.posterior(xShow).argmax(dim=1)

print("Label:", yShow.cpu().tolist())
print("Pred :", pred.cpu().tolist())